# NLP-Based Detection of Manipulative UX Patterns in Web Interfaces
## Analysis Notebook - Katarzyna Stępień, Uniwersytet SWPS, Warszawa 2026

This notebook documents the full NLP analysis pipeline accompanying the bachelor's thesis:
- **Dataset:** Mathur et al. (2019) dark patterns corpus, supplemented via Kaggle (`uppal2023darkpatterns`)
- **Task:** Binary classification (manipulative vs. non-manipulative) + multiclass category labelling
- **Primary model:** Logistic Regression with TF-IDF bigram features
- **Key results:** F1 = 0.936, ROC AUC = 0.976 (binary); overall accuracy = 0.909 (multiclass)

---


## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support
)

# ── Plot style ──────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize':  (10, 6),
    'font.size':       11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'figure.dpi':      120,
})

PALETTE = {
    'dark':     '#C0392B',
    'not_dark': '#2980B9',
    'neutral':  '#7F8C8D',
    'accent':   '#F39C12',
}

print("Libraries loaded.")


## 2. Data Loading

Dataset sourced from Mathur et al. (2019) and retrieved via Kaggle (`uppal2023darkpatterns`).
Each row contains:
- `text` — short UX copy segment extracted from an e-commerce page
- `label` — binary class (1 = manipulative, 0 = non-manipulative)
- `Pattern Category` — multiclass influence mechanism label


In [ ]:
df = pd.read_csv('dataset.csv')

print(f"Dataset shape:  {df.shape}")
print(f"Columns:        {df.columns.tolist()}")
print(f"Missing values:\n{df.isnull().sum()}")
df.head(8)


## 3. Exploratory Data Analysis

### 3.1 Binary Label Distribution

In [ ]:
label_counts = df['label'].value_counts()
label_names  = {0: 'Not Dark Pattern', 1: 'Dark Pattern'}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    [label_names[k] for k in label_counts.index],
    label_counts.values,
    color=[PALETTE['dark'], PALETTE['not_dark']],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, val in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{val:,}\n({val/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Count')
ax.set_title('Binary Label Distribution', fontweight='bold', pad=12)
ax.set_ylim(0, label_counts.max() * 1.2)
plt.tight_layout()
plt.savefig('viz_01_binary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(label_counts.rename(index=label_names).to_string())


### 3.2 Dark Pattern Category Distribution

In [ ]:
category_counts = (
    df[df['label'] == 1]['Pattern Category']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Category', 'Pattern Category': 'Count'})
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(category_counts['Pattern Category'], category_counts['Count'],
               color=PALETTE['dark'], edgecolor='white', linewidth=1)
for bar, val in zip(bars, category_counts['Count']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Count')
ax.set_title('Dark Pattern Category Distribution', fontweight='bold', pad=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('viz_02_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Text Length Distribution

In [ ]:
df['char_count'] = df['text'].astype(str).str.len()
df['word_count'] = df['text'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, label in zip(axes,
                           ['char_count', 'word_count'],
                           ['Character count', 'Word count']):
    for lbl, color, name in [(1, PALETTE['dark'], 'Dark Pattern'),
                              (0, PALETTE['not_dark'], 'Not Dark Pattern')]:
        ax.hist(df[df['label']==lbl][col], bins=40, alpha=0.6,
                color=color, label=name, edgecolor='none')
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
ax.set_title('Word Count by Class', fontweight='bold')
axes[0].set_title('Character Count by Class', fontweight='bold')
plt.tight_layout()
plt.savefig('viz_03_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('label')[['char_count','word_count']].describe().round(1))


### 3.4 Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lbl, title, bg, colormap in [
    (axes[0], 1, 'Dark Pattern Vocabulary',     '#1a1a1a', 'Reds'),
    (axes[1], 0, 'Non-Manipulative Vocabulary', '#f9f9f9', 'Blues'),
]:
    text = ' '.join(df[df['label'] == lbl]['text'].astype(str))
    wc = WordCloud(width=700, height=400, background_color=bg,
                   colormap=colormap, max_words=120,
                   collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontweight='bold', fontsize=13, pad=10)

plt.tight_layout()
plt.savefig('viz_04_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Preprocessing and TF-IDF Vectorisation

Pipeline steps:
1. Lowercase conversion and whitespace normalisation
2. Stop word removal (NLTK English corpus; negation markers retained)
3. TF-IDF vectorisation with unigrams and bigrams (`ngram_range=(1,2)`)
4. Vocabulary capped at 5,000 terms; `min_df=2`, `max_df=0.95`
5. Stratified 80/20 train/test split (`random_state=42`)


In [ ]:
df['text_clean'] = df['text'].astype(str).str.lower().str.strip()

X = df['text_clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.95
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Train size:           {len(X_train):,}  ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test size:            {len(X_test):,}  ({len(X_test)/len(df)*100:.0f}%)")
print(f"TF-IDF matrix shape:  {X_train_tfidf.shape}")
print(f"Vocabulary size:      {len(tfidf.get_feature_names_out()):,}")


## 5. Binary Classification

Five models trained and evaluated on the held-out test set.
Primary ranking metric: **F1-score** (harmonic mean of precision and recall).
5-fold stratified cross-validation performed on the training set.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (Linear)':        LinearSVC(random_state=42, max_iter=2000),
    'Naive Bayes':         MultinomialNB(),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    cv     = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring='f1')

    if hasattr(model, 'predict_proba'):
        auc = roc_auc_score(y_test, model.predict_proba(X_test_tfidf)[:, 1])
    elif hasattr(model, 'decision_function'):
        auc = roc_auc_score(y_test, model.decision_function(X_test_tfidf))
    else:
        auc = None

    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC':       auc,
        'CV F1':     cv.mean(),
        'CV SD':     cv.std(),
    })

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.round(3).to_string(index=False))


### 5.1 Model Comparison Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors_bar = [PALETTE['not_dark'], PALETTE['accent'], PALETTE['neutral'], PALETTE['dark']]
x = np.arange(len(results_df))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
for i, (metric, color) in enumerate(zip(metrics, colors_bar)):
    ax.bar(x + i*width, results_df[metric], width,
           label=metric, color=color, edgecolor='white', linewidth=0.8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Binary Classification — Model Comparison', fontweight='bold', pad=12)
ax.set_ylim(0.70, 1.02)
ax.axhline(0.9, color='grey', lw=1, ls='--', alpha=0.6, label='0.90 reference')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('viz_05_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.2 Best Model - Logistic Regression

In [ ]:
best_model = LogisticRegression(max_iter=1000, random_state=42)
best_model.fit(X_train_tfidf, y_train)
y_pred_best  = best_model.predict(X_test_tfidf)
y_proba_best = best_model.predict_proba(X_test_tfidf)[:, 1]

print(classification_report(
    y_test, y_pred_best,
    target_names=['Not Dark Pattern', 'Dark Pattern']
))


### 5.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Dark', 'Dark Pattern'],
            yticklabels=['Not Dark', 'Dark Pattern'],
            annot_kws={'size': 16, 'fontweight': 'bold'},
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_ylabel('True label', fontsize=11)
ax.set_title('Confusion Matrix — Logistic Regression', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('viz_06_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"True Negatives:  {tn}  |  False Positives: {fp}")
print(f"False Negatives: {fn}  |  True Positives:  {tp}")


### 5.4 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_score   = roc_auc_score(y_test, y_proba_best)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color=PALETTE['dark'], lw=2.5,
        label=f'Logistic Regression (AUC = {auc_score:.3f})')
ax.fill_between(fpr, tpr, alpha=0.08, color=PALETTE['dark'])
ax.plot([0,1],[0,1], color=PALETTE['neutral'], lw=1.5, ls='--',
        label='Random classifier (AUC = 0.500)')

# Mark operating point
op_idx = np.argmin(np.abs(fpr - 0.025))
ax.scatter(fpr[op_idx], tpr[op_idx], color=PALETTE['dark'], s=80, zorder=5,
           label=f'Operating point (FPR={fpr[op_idx]:.3f}, TPR={tpr[op_idx]:.3f})')

ax.set_xlim([0, 1]); ax.set_ylim([0, 1.03])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression', fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('viz_07_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.5 Feature Importance

In [ ]:
feature_names = tfidf.get_feature_names_out()
coef          = best_model.coef_[0]
feat_df       = pd.DataFrame({'feature': feature_names, 'weight': coef})

top_pos = feat_df.nlargest(15, 'weight')
top_neg = feat_df.nsmallest(15, 'weight').sort_values('weight')

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, data, color, title in [
    (axes[0], top_pos, PALETTE['dark'],     'Top 15 — Dark Pattern indicators'),
    (axes[1], top_neg, PALETTE['not_dark'], 'Top 15 — Non-manipulative indicators'),
]:
    bars = ax.barh(data['feature'], data['weight'],
                   color=color, edgecolor='white', linewidth=0.6)
    ax.set_xlabel('TF-IDF coefficient', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11, pad=8)
    ax.invert_yaxis()
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Logistic Regression — Feature Importance', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

feat_df.sort_values('weight', ascending=False).to_csv('feature_importance.csv', index=False)


## 6. Multiclass Classification

The same TF-IDF pipeline applied to a 8-class prediction task
(7 dark pattern categories + *Not Dark Pattern*).
Class imbalance is severe for rare categories (Forced Action: n=4, Sneaking: n=12).


In [ ]:
y_multi = df['Pattern Category']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

tfidf_m = TfidfVectorizer(
    max_features=5000, ngram_range=(1,2),
    stop_words='english', min_df=2, max_df=0.95
)
X_train_m_tfidf = tfidf_m.fit_transform(X_train_m)
X_test_m_tfidf  = tfidf_m.transform(X_test_m)

multi_model = LogisticRegression(max_iter=1000, random_state=42)
multi_model.fit(X_train_m_tfidf, y_train_m)
y_pred_multi = multi_model.predict(X_test_m_tfidf)

print(classification_report(y_test_m, y_pred_multi))
print(f"Overall accuracy: {accuracy_score(y_test_m, y_pred_multi):.4f}")


### 6.1 Multiclass Confusion Matrix

In [ ]:
labels_m  = sorted(y_multi.unique())
cm_multi  = confusion_matrix(y_test_m, y_pred_multi, labels=labels_m)
cm_norm   = cm_multi.astype(float) / cm_multi.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=cm_multi, fmt='d', cmap='YlOrRd',
            xticklabels=labels_m, yticklabels=labels_m,
            linewidths=0.4, linecolor='white',
            annot_kws={'size': 9}, vmin=0, vmax=1, ax=ax)
ax.set_xlabel('Predicted category', fontsize=11)
ax.set_ylabel('True category', fontsize=11)
ax.set_title('Multiclass Confusion Matrix (raw counts; colour = row-normalised)',
             fontweight='bold', pad=12)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('viz_09_multiclass_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


### 6.2 Per-Category Metrics

In [ ]:
precision_m, recall_m, f1_m, support_m = precision_recall_fscore_support(
    y_test_m, y_pred_multi, labels=labels_m
)
cat_df = pd.DataFrame({
    'Category':  labels_m,
    'Precision': precision_m,
    'Recall':    recall_m,
    'F1':        f1_m,
    'Support':   support_m,
}).sort_values('F1', ascending=False).reset_index(drop=True)

print(cat_df.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(cat_df))
w = 0.25
ax.bar(x - w,   cat_df['Precision'], w, label='Precision', color=PALETTE['not_dark'], edgecolor='white')
ax.bar(x,       cat_df['Recall'],    w, label='Recall',    color=PALETTE['neutral'],  edgecolor='white')
ax.bar(x + w,   cat_df['F1'],        w, label='F1',        color=PALETTE['dark'],     edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(cat_df['Category'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.12)
ax.set_title('Per-Category Classification Metrics', fontweight='bold', pad=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('viz_10_category_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Summary

In [ ]:
best = results_df.iloc[0]
print("=" * 65)
print("  RESULTS SUMMARY")
print("=" * 65)

print("\n  DATASET")
print(f"    Samples total:        {len(df):,}")
print(f"    Dark patterns:        {(df['label']==1).sum():,}  (50%)")
print(f"    Non-manipulative:     {(df['label']==0).sum():,}  (50%)")
print(f"    Pattern categories:   {df['Pattern Category'].nunique()}")

print("\n  PREPROCESSING")
print("    Lowercase + whitespace normalisation")
print("    Stop word removal (English, negation markers retained)")
print("    TF-IDF: ngram_range=(1,2), max_features=5000, min_df=2, max_df=0.95")
print("    Split: 80% train / 20% test, stratified")

print(f"\n  BEST MODEL: {best['Model']}")
print(f"    Accuracy:   {best['Accuracy']:.3f}")
print(f"    Precision:  {best['Precision']:.3f}")
print(f"    Recall:     {best['Recall']:.3f}")
print(f"    F1:         {best['F1']:.3f}")
print(f"    ROC AUC:    {best['AUC']:.3f}")
print(f"    CV F1:      {best['CV F1']:.3f}  (SD = {best['CV SD']:.3f})")

print("\n  KEY FEATURES (dark pattern indicators)")
top10 = feat_df.nlargest(10, 'weight')['feature'].tolist()
print(f"    {', '.join(top10)}")

print(f"\n  MULTICLASS ACCURACY: {accuracy_score(y_test_m, y_pred_multi):.3f}")
print("    Best categories:  Scarcity, Social Proof, Not Dark Pattern")
print("    Unlearnable:      Forced Action (n=1 test), Sneaking (n=2 test)")

print("\n  LIMITATIONS")
print("    Text-only detection — visual/structural dark patterns not covered")
print("    Dataset reflects 2019 English-language e-commerce conventions")
print("    Rare categories require additional labelled data")
print("=" * 65)
